# Essentials AI · Applications de l'IA et automatisation

**Objectif :** Automatisation avec n8n et génération augmentée par récupération (RAG).

Lance ce notebook depuis le dossier `notebooks/`.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath("../labs"))   # brewly, fakeapi
os.chdir("../data")                               # penguins.csv, titanic.csv, tips.csv

---
## Automatiser avec n8n et l'IA

**n8n** est un outil d'automatisation visuel et open source : on relie des **nœuds** (*nodes*) dans un **workflow**, sans ou avec peu de code. L'Essentials y consacre une journée (pipeline de contenu, moteur de campagne GreenWatch, RAG dans n8n). C'est l'outil idéal pour prototyper en une heure un système qui combine des API, un LLM et des données.

## Les notions clés

- **Déclencheur** (*trigger*) : ce qui lance le workflow. Un horaire (Schedule), un webhook (une URL appelée par un autre service), un nouveau mail, un formulaire.
- **Nœuds** : chaque étape. HTTP Request, Google Sheets, Slack, IF, Merge, Code, et des nœuds IA (AI Agent, chat models, bases vectorielles).
- **Items** : les données circulent sous forme d'une **liste d'éléments**, chacun avec un champ `json`. Chaque nœud traite tous les items qu'il reçoit.
- **Expressions** : `{{ $json.titre }}` insère un champ de l'item courant dans un paramètre.
- **Identifiants** (*credentials*) : les clés d'API sont stockées à part, jamais en clair dans le workflow.

*Animation interactive « n8n » : à voir sur la plateforme web.*

## Le format des items

Un nœud **Code** de n8n reçoit et renvoie ce format. En Python, ça donne :

In [ ]:
items = [
    {"json": {"titre": "Lancement du produit", "vues": 1200, "langue": "fr"}},
    {"json": {"titre": "Product launch", "vues": 300, "langue": "en"}},
    {"json": {"titre": "Bilan trimestriel", "vues": 4500, "langue": "fr"}},
]
# Ce que ferait un nœud IF puis un nœud Set : garder le français, ajouter un champ
sortie = [{"json": {**it["json"], "populaire": it["json"]["vues"] > 1000}} for it in items if it["json"]["langue"] == "fr"]
sortie

## Installer n8n sur ton ordinateur

```bash
# Avec Node.js installé :
npx n8n
# ou avec Docker :
docker run -it --rm -p 5678:5678 -v n8n_data:/home/node/.n8n docker.n8n.io/n8nio/n8n
```

Puis ouvre http://localhost:5678. Il existe aussi une version hébergée payante (n8n Cloud), avec un essai gratuit.

> **À retenir** : n8n pour prototyper vite et automatiser des processus métier ; Python quand il faut une logique complexe, des tests et du contrôle fin. Les deux se combinent : un nœud HTTP Request peut appeler ton API FastAPI.

**À faire :**

- [ ] Installer n8n en local (npx n8n ou Docker) et ouvrir http://localhost:5678
- [ ] Créer un workflow : déclencheur manuel → HTTP Request vers une API publique → nœud Code qui filtre les items
- [ ] Ajouter un nœud de modèle de langage (avec ta clé d'API) qui résume chaque item
- [ ] Envoyer le résultat dans un Google Sheet ou par e-mail
- [ ] Exporter le workflow en JSON et le publier sur GitHub

### Exercice 1 · Un nœud Code en Python

Écris `noeud_code(items)` qui imite un nœud Code de n8n : garde les items dont `statut` vaut `nouveau` et dont `montant` dépasse 100, et renvoie pour chacun un item `{"json": {"client": ..., "montant_ttc": ..., "priorite": ...}}` où `montant_ttc` vaut montant × 1,2 arrondi à 2 décimales, et `priorite` vaut `haute` au-delà de 1 000 € TTC, `normale` sinon.

<details><summary>Indice</summary>

Parcours les items, lis it["json"], filtre, puis construis un nouveau {"json": {...}} pour chaque item gardé.
</details>

In [ ]:
items = [
    {"json": {"client": "Ada", "montant": 900, "statut": "nouveau"}},
    {"json": {"client": "Linus", "montant": 80, "statut": "nouveau"}},
    {"json": {"client": "Grace", "montant": 2500, "statut": "traité"}},
    {"json": {"client": "Alan", "montant": 150, "statut": "nouveau"}},
]

def noeud_code(items):
    pass

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
assert noeud_code(items) == [{"json": {"client": "Ada", "montant_ttc": 1080.0, "priorite": "haute"}}, {"json": {"client": "Alan", "montant_ttc": 180.0, "priorite": "normale"}}], f"noeud_code renvoie {noeud_code(items)}."
print("Bravo, tout est correct.")

---
## Découvrir l'IA générative et le RAG

Une grande partie de la formation porte sur l'**IA générative** : Transformers, RAG, agents. Voici les notions de base pour ne pas être perdu.

## Ce qu'est un grand modèle de langage

Un **LLM** (*Large Language Model*) est un réseau de neurones de type **Transformer**, entraîné à prédire le **token** suivant d'un texte. Un token est un morceau de mot : en moyenne, environ 4 caractères en anglais.

- Le **prompt** est le texte d'entrée. Le **prompt système** fixe le rôle et les règles du modèle.
- La **fenêtre de contexte** est la quantité maximale de tokens que le modèle peut lire en une fois.
- La **température** règle la part de hasard : basse pour des réponses stables, plus haute pour de la créativité.
- Un modèle peut **halluciner** : produire une réponse fausse mais crédible.

## Appeler un modèle depuis Python

Les fournisseurs proposent une API et un SDK Python. Le principe est toujours le même : une clé d'API, un modèle, une liste de messages. Ce code est à exécuter sur ton ordinateur, avec ta propre clé dans un fichier `.env`.

```python
# pip install anthropic python-dotenv
import os
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()                                   # lit ANTHROPIC_API_KEY dans .env
client = Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

reponse = client.messages.create(
    model="claude-sonnet-5-5",
    max_tokens=300,
    system="Tu es un tuteur de data science. Réponds en trois phrases maximum.",
    messages=[{"role": "user", "content": "Explique le surapprentissage à un débutant."}],
)
print(reponse.content[0].text)
```

Les SDK d'OpenAI, de Mistral ou de Google fonctionnent sur le même modèle.

## Le RAG : donner des documents au modèle

Un LLM ne connaît pas tes documents internes, et ses connaissances s'arrêtent à une date. Le **RAG** (*Retrieval-Augmented Generation*) résout ce problème en trois étapes :

1. **Indexer** : découper les documents en morceaux (*chunks*) et transformer chacun en vecteur (**embedding**), stocké dans une **base vectorielle**.
2. **Retrouver** : transformer la question en vecteur et chercher les morceaux les plus proches, par **similarité cosinus** (semaine 3).
3. **Générer** : envoyer au LLM la question accompagnée des morceaux retrouvés, avec la consigne de répondre à partir d'eux.

*Animation interactive « rag » : à voir sur la plateforme web.*

Voici l'étape 2 avec un embedding très simplifié : un vecteur de comptage de mots. Les vrais embeddings capturent le **sens**, mais le principe de recherche est identique.

In [ ]:
import numpy as np

documents = [
    "La descente de gradient minimise la fonction de perte en suivant le gradient.",
    "pandas permet de charger un fichier CSV dans un DataFrame.",
    "Le surapprentissage arrive quand le modèle apprend le bruit des données d'entraînement.",
]
vocab = sorted({m.strip(".,").lower() for d in documents for m in d.split()})

def vectoriser(texte):
    mots = [m.strip(".,?").lower() for m in texte.split()]
    return np.array([mots.count(v) for v in vocab], dtype=float)

def cosinus(u, v):
    n = np.linalg.norm(u) * np.linalg.norm(v)
    return 0.0 if n == 0 else u @ v / n

question = "Comment charger un CSV avec pandas ?"
q = vectoriser(question)
scores = [cosinus(q, vectoriser(d)) for d in documents]
print([round(s, 2) for s in scores])
print("Document retrouvé :", documents[int(np.argmax(scores))])

> **À retenir** : un agent IA est un LLM qui peut appeler des **outils** (une recherche, une base de données, du code) et enchaîner plusieurs étapes. Tu en construiras pendant la formation.

### Exercice 1 · Construire le prompt d'un RAG

Écris `construire_prompt(question, documents, k=2)` qui retrouve les `k` documents les plus similaires à la question (fonctions `vectoriser` et `cosinus` fournies), puis renvoie un texte qui contient : la ligne `Contexte :`, les documents retenus (un par ligne, **du plus au moins similaire**), puis la ligne `Question : ` suivie de la question.

<details><summary>Indice</summary>

sorted(documents, key=lambda d: cosinus(q, vectoriser(d)), reverse=True)[:k], puis assemble le texte avec "\n".join(...).
</details>

In [ ]:
import numpy as np

documents = [
    "Le RAG combine une recherche de documents et un modèle de langage.",
    "Une base vectorielle stocke des embeddings pour la recherche par similarité.",
    "Les crêpes se préparent avec de la farine, des œufs et du lait.",
    "Un embedding est un vecteur qui représente le sens d'un texte.",
]
vocab = sorted({m.strip(".,").lower() for d in documents for m in d.split()})

def vectoriser(texte):
    mots = [m.strip(".,?").lower() for m in texte.split()]
    return np.array([mots.count(v) for v in vocab], dtype=float)

def cosinus(u, v):
    n = np.linalg.norm(u) * np.linalg.norm(v)
    return 0.0 if n == 0 else u @ v / n

def construire_prompt(question, documents, k=2):
    pass

print(construire_prompt("Qu'est-ce qu'un embedding dans une base vectorielle ?", documents))

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
p = construire_prompt("Qu'est-ce qu'un embedding dans une base vectorielle ?", documents)
assert isinstance(p, str), "construire_prompt doit renvoyer un texte (as-tu oublié return ?)."
assert p.startswith("Contexte :"), "Le prompt doit commencer par « Contexte : »."
assert "Question : Qu'est-ce qu'un embedding" in p, "Le prompt doit se terminer par « Question : » suivi de la question."
assert "crêpes" not in p, "Le document sur les crêpes ne devrait pas être retenu."
assert p.index("Une base vectorielle") < p.index("Un embedding est"), "Les documents doivent être classés du plus au moins similaire."
assert "Le RAG combine" in construire_prompt("le RAG combine recherche et modèle", documents, k=1), "Avec k=1, seul le meilleur document doit apparaître."
print("Bravo, tout est correct.")

---
## Un RAG de A à Z en Python

Tu as vu le principe du RAG : retrouver les bons morceaux de documents, puis demander au LLM de répondre **à partir d'eux**. On le construit maintenant de bout en bout, sur la base de connaissances du support de **CloudSync**, comme dans le projet « Créez le CloudSync RAG » de l'Essentials.

In [ ]:
from cloudsync import DOCS, QUESTIONS_TEST
print(len(DOCS), "documents. Exemple :")
print(DOCS[0]["titre"], "→", DOCS[0]["texte"][:120], "…")

## 1. Découper (chunking)

Les documents sont découpés en **morceaux** (*chunks*) assez petits pour être précis, assez grands pour garder le sens. Un **chevauchement** entre morceaux évite de couper une information en deux.

*Animation interactive « chunk » : à voir sur la plateforme web.*

## 2. Vectoriser

Chaque morceau devient un vecteur. En production, on utilise un **modèle d'embeddings** qui capture le sens (deux phrases synonymes ont des vecteurs proches). Ici, on utilise **TF-IDF**, une méthode classique fondée sur les mots, disponible dans scikit-learn : le principe de la recherche est exactement le même.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

textes = [d["titre"] + ". " + d["texte"] for d in DOCS]
vectoriseur = TfidfVectorizer()
matrice = vectoriseur.fit_transform(textes)        # une ligne par document
print(matrice.shape, "→ documents × mots du vocabulaire")

## 3. Retrouver

In [ ]:
def rechercher(question, k=2):
    q = vectoriseur.transform([question])
    scores = cosine_similarity(q, matrice)[0]
    meilleurs = scores.argsort()[::-1][:k]
    return [(DOCS[i]["id"], round(float(scores[i]), 3)) for i in meilleurs]

rechercher("Combien de temps restent les fichiers supprimés ?")

## 4. Générer, avec les sources

Le prompt contient les morceaux retrouvés, numérotés, et une consigne stricte : répondre uniquement à partir d'eux, citer les sources, et dire « je ne sais pas » sinon.

In [ ]:
import llm

async def repondre(question, k=2):            # async : la fonction attend le LLM
    ids = [i for i, _ in rechercher(question, k)]
    contexte = "\n".join(f"[{i}] {next(d['texte'] for d in DOCS if d['id'] == i)}" for i in ids)
    prompt = f"""Tu es l'assistant du support CloudSync. Réponds en français, en 2 phrases maximum,
uniquement à partir des documents ci-dessous, et cite la source entre crochets.
Si la réponse n'y est pas, réponds : « Je ne sais pas, je transmets ta question à un conseiller. »

Documents :
{contexte}

Question : {question}"""
    return await llm.complete(prompt), ids

texte, sources = await repondre("Puis-je protéger un lien de partage par un mot de passe avec l'offre gratuite ?")
print(texte, "\nSources :", sources)

## 5. Évaluer la recherche

Si la recherche rate le bon document, le LLM ne peut pas bien répondre. On mesure donc d'abord le **rappel@k** : la proportion de questions pour lesquelles un document pertinent figure parmi les k premiers résultats.

In [ ]:
for k in [1, 2, 3]:
    trouves = sum(any(i in q["pertinents"] for i, _ in rechercher(q["question"], k)) for q in QUESTIONS_TEST)
    print(f"rappel@{k} = {trouves}/{len(QUESTIONS_TEST)}")

> **À retenir** : un RAG, c'est découper, vectoriser, retrouver, générer avec les sources, et évaluer chaque étape séparément. La qualité de la recherche limite la qualité de la réponse.

### Exercice 1 · Découper avec chevauchement

Écris `decouper(texte, taille, chevauchement)` qui découpe le texte en morceaux de `taille` **mots**, chaque morceau commençant `taille - chevauchement` mots après le précédent. Le dernier morceau peut être plus court ; ne crée pas de morceau qui ne contiendrait que des mots déjà vus. Lève une `ValueError` si `chevauchement >= taille`.

<details><summary>Indice</summary>

Le pas vaut taille - chevauchement. Arrête-toi dès qu'un morceau atteint la fin du texte.
</details>

In [ ]:
def decouper(texte, taille, chevauchement):
    pass

print(decouper("un deux trois quatre cinq six sept", 3, 1))

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
assert decouper("un deux trois quatre cinq six sept", 3, 1) == ["un deux trois", "trois quatre cinq", "cinq six sept"], f"Résultat : {decouper('un deux trois quatre cinq six sept', 3, 1)}."
assert decouper("a b c d e f g h", 4, 2) == ["a b c d", "c d e f", "e f g h"], f"Pour 8 mots, taille 4 et chevauchement 2 : {decouper('a b c d e f g h', 4, 2)}."
assert decouper("a b c d e", 3, 0) == ["a b c", "d e"], "Sans chevauchement, le dernier morceau peut être plus court."
try:
    decouper("a b", 2, 2)
    assert False, "Un chevauchement >= taille doit lever une ValueError."
except ValueError:
    pass
print("Bravo, tout est correct.")

### Exercice 2 · Un moteur de recherche réutilisable

Écris une classe `Recherche` avec `fit(docs)` (une liste de dictionnaires avec `id`, `titre`, `texte` ; elle vectorise `titre + ". " + texte` avec `TfidfVectorizer` et renvoie `self`) et `chercher(question, k=2)` qui renvoie la liste des `id` des k documents les plus similaires, du plus au moins similaire.

<details><summary>Indice</summary>

Inspire-toi de la leçon : fit_transform dans fit, transform puis cosine_similarity dans chercher, argsort()[::-1][:k].
</details>

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from cloudsync import DOCS

class Recherche:
    pass

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
r = Recherche()
assert r.fit(DOCS) is r, "fit doit renvoyer self."
assert r.chercher("Comment récupérer un fichier supprimé dans la corbeille ?", 1) == ["corbeille"], f"Résultat : {r.chercher('Comment récupérer un fichier supprimé dans la corbeille ?', 1)}."
res = r.chercher("prix de l'offre Pro par mois", 3)
assert len(res) == 3 and res[0] == "tarifs", f"Pour une question de prix, le premier résultat doit être tarifs ; tu obtiens {res}."
print("Bravo, tout est correct.")

### Exercice 3 · Mesurer le rappel@k

Écris `rappel_at_k(recherche, questions, k)` qui renvoie la proportion de questions (dictionnaires avec `question` et `pertinents`) pour lesquelles **au moins un** document pertinent figure dans les k premiers résultats de `recherche(question, k)`. Puis calcule `r1` et `r3` pour la fonction de recherche fournie.

<details><summary>Indice</summary>

any(p in resultats for p in q["pertinents"]) vaut True si au moins un document pertinent est retrouvé.
</details>

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from cloudsync import DOCS, QUESTIONS_TEST

vect = TfidfVectorizer()
mat = vect.fit_transform([d["titre"] + ". " + d["texte"] for d in DOCS])

def recherche(question, k):
    s = cosine_similarity(vect.transform([question]), mat)[0]
    return [DOCS[i]["id"] for i in s.argsort()[::-1][:k]]

def rappel_at_k(recherche, questions, k):
    pass

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
fake = lambda q, k: ["a", "b", "c"][:k]
qs = [{"question": "x", "pertinents": ["b"]}, {"question": "y", "pertinents": ["z"]}]
assert rappel_at_k(fake, qs, 1) == 0 and rappel_at_k(fake, qs, 2) == 0.5, "Teste ta fonction sur un cas simple : avec k=2, une question sur deux est trouvée."
assert 0 < r1 <= r3 <= 1, f"r1 = {r1}, r3 = {r3} : le rappel ne peut qu'augmenter avec k."
assert r3 == rappel_at_k(recherche, QUESTIONS_TEST, 3), "r3 doit être calculé avec ta fonction et k = 3."
print("Bravo, tout est correct.")